<a href="https://colab.research.google.com/github/shahrzadtrb-co/Privacy-Preserving-Techniques-for-LLM-Code-Completion/blob/main/Privacy_Preserving_Techniques_for_LLM_Code_Completion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install -q datasets "transformers<5" evaluate python-Levenshtein matplotlib

In [33]:
#library imports
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import Levenshtein
import matplotlib.pyplot as plt
import re

In [34]:
#load the test split of HumanEval
dataset = load_dataset("openai/openai_humaneval", split="test")
#extract rows 0 till 19
dataset_20 = dataset.select(range(20))
#confrim the correctness
print(len(dataset_20))
print(dataset_20[0].keys())
print(dataset_20[0]["prompt"][:300])
print(dataset_20[0]["canonical_solution"][:300])

20
dict_keys(['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'])
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False



In [35]:
#verify by printing the first 3 examples
for i in range(3):
    print("\nExample", i)
    print("Task ID:", dataset_20[i]["task_id"])
    print("Prompt:\n", dataset_20[i]["prompt"][:250])
    print("Canonical solution:\n", dataset_20[i]["canonical_solution"][:250])


Example 0
Task ID: HumanEval/0
Prompt:
 from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0]
Canonical solution:
     for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return Fals

Example 1
Task ID: HumanEval/1
Prompt:
 from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
    separate those group into separate strings and retur
Canonical solution:
     result = []
    current_string = []
    current_depth = 0

    for c in paren_string:
        if c == '(':
            

In [36]:
# Obfuscation with Python's tokenize module: only real identifiers are renamed,
# and docstring words stay readable (earlier regex version also scrambled docstrings)
import builtins
import io
import keyword
import re
import tokenize

BUILTIN_NAMES = set(dir(builtins))


def _import_names(code: str):
    """Module names and imported names (e.g. typing, List), which are kept as they are."""
    names = set()
    for line in code.splitlines():
        line = line.strip()
        m = re.match(r"from\s+([\w.]+)\s+import\s+(.+)", line)
        if m:
            names.update(m.group(1).split("."))
            for part in m.group(2).replace("(", "").replace(")", "").split(","):
                if part.strip():
                    names.add(part.split(" as ")[-1].strip())
            continue
        m = re.match(r"import\s+(.+)", line)
        if m:
            for part in m.group(1).split(","):
                names.update(part.strip().split(" as ")[-1].strip().split("."))
    return names


def _tokens(code: str):
    return list(tokenize.generate_tokens(io.StringIO(code).readline))


def low_obfuscation(code: str, keep_builtins: bool = True, rename_in_text: bool = True):
    #Rename the identifiers defined in the prompt (function, parameter,Rename the identifiers defined in the prompt (function, parameter and variable names).

    if not code:
        return ""
    keep = set(keyword.kwlist) | set(keyword.softkwlist)
    if keep_builtins:
        keep |= BUILTIN_NAMES | _import_names(code)

    try:
        toks = _tokens(code)
    except (tokenize.TokenError, IndentationError, SyntaxError) as e:
        print("tokenize failed, prompt left unchanged:", e)
        return code

    # Pass 1: decide the new name for every identifier in the code
    mapping = {}
    for tok in toks:
        if tok.type == tokenize.NAME and tok.string not in keep and len(tok.string) > 2:
            mapping.setdefault(tok.string, "v" + str(len(mapping)))

    # Whole-word pattern for the mapped names, longest first
    name_pattern = None
    if mapping:
        alternatives = "|".join(re.escape(n) for n in sorted(mapping, key=len, reverse=True))
        name_pattern = re.compile(r"\b(" + alternatives + r")\b")

    # Pass 2: rewrite the tokens
    out = []
    for tok in toks:
        text = tok.string
        if tok.type == tokenize.NAME and text in mapping:
            text = mapping[text]
        elif rename_in_text and name_pattern and tok.type in (tokenize.STRING, tokenize.COMMENT):
            text = name_pattern.sub(lambda m: mapping[m.group(0)], text)
        out.append((tok.type, text, tok.start, tok.end, tok.line))
    return tokenize.untokenize(out)


def high_obfuscation(code: str, keep_builtins: bool = True, rename_in_text: bool = True):
    """Low obfuscation, plus comments removed and every string (docstrings too) replaced by "STR"."""
    if not code:
        return ""
    s = low_obfuscation(code, keep_builtins, rename_in_text)
    try:
        toks = _tokens(s)
    except (tokenize.TokenError, IndentationError, SyntaxError) as e:
        print("tokenize failed, prompt left unchanged:", e)
        return s
    out = []
    for tok in toks:
        if tok.type == tokenize.COMMENT:
            continue
        text = '"STR"' if tok.type == tokenize.STRING else tok.string
        out.append((tok.type, text, tok.start, tok.end, tok.line))
    s = tokenize.untokenize(out)
    s = "\n".join(line.rstrip() for line in s.splitlines()) + "\n"
    return re.sub(r"\n\s*\n\s*\n+", "\n\n", s)


In [37]:
#prompt from first 20 sample
sample = dataset_20[0]["prompt"]

print("Type of sample:", type(sample))
#irst 200 characters of the original prompt
print("\nOriginal:\n", sample[:200])
print("\nLow:\n", low_obfuscation(sample)[:200])
print("\nHigh:\n", high_obfuscation(sample)[:200])


Type of sample: <class 'str'>

Original:
 from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given thr

Low:
 from typing import List


def v0(v1: List[float], v2: float) -> bool:
    """ Check if in given list of v1, are any two v1 closer to each other than
    given v2.
    >>> v0([1.0, 2.0, 3.0], 0.5)
    

High:
 from typing import List

def v0(v1: List[float], v2: float) -> bool:
    "STR"



In [38]:
# torch is needed as the model backend (preinstalled on Colab)
# !pip install torch


In [39]:
#Loading CodeT5-small
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Salesforce/codet5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [40]:
#function to give a prompt to CodeT5-small and get the model’s predicted code
def generate_completion(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    ids = model.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(ids[0], skip_special_tokens=True)


In [41]:
#testing the model
print(generate_completion(dataset_20[0]["prompt"])[:300])


has_close_elements([1.0, 2.8, 3.0,0.5)
   0.5)
   0.5)
   0.5)
   0.5)
   0.5)
   


In [ ]:
#generate all completions
results = []

for i, item in enumerate(dataset_20):
    base = item["prompt"]
    ref = item["canonical_solution"]

    prompts = {
        "original": base,
        "low": low_obfuscation(base),
        "high": high_obfuscation(base),
    }

    for level, p in prompts.items():
        comp = generate_completion(p)
        results.append({
            "i": i,
            "level": level,
            "prompt": p,
            "base": base,
            "comp": comp,
            "ref": ref,
        })

print("Total completions:", len(results))


In [ ]:
import Levenshtein
import matplotlib.pyplot as plt


#Compute length of the longest common subsequence between two token lists
def lcs_length(a_tokens, b_tokens):

    m = len(a_tokens)
    n = len(b_tokens)

    # dp[i][j] = LCS length for prefixes a[:i], b[:j]
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if a_tokens[i] == b_tokens[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


# Compute the Longest Common Subsequence (LCS) length between two strings
def lcs_char(a, b):
    m, n = len(a), len(b)
    # Create a DP table of size (m+1) x (n+1)
    # dp[i][j] stores the LCS length for a[:i] and b[:j]
    dp = [[0]*(n+1) for _ in range(m+1)]

    # Fill the DP table
    for i in range(m):
        for j in range(n):
            if a[i] == b[j]:
                # Characters match, extend the LCS
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])
    # full LCS length
    return dp[m][n]

#character-level ROUGE-L style F1 utility score
def utility(pred, ref):
    a = pred
    b = ref
    # If either is empty, utility is zero
    if not a or not b:
        return 0.0

    lcs = lcs_char(a, b)
    # Precision and recall based on character overlap
    precision = lcs / len(a)
    recall = lcs / len(b)

    if precision + recall == 0:
        return 0.0

    #F1 score
    return 2 * precision * recall / (precision + recall)


# Return a score between 0 and 1 Higher means more different from the original

def privacy(obfuscated_prompt, base_prompt):

    d = Levenshtein.distance(obfuscated_prompt, base_prompt)
    m = max(len(obfuscated_prompt), len(base_prompt))

    if m == 0:
        return 0.0

    return d / m


#privacy and utility for all 60 results
for r in results:
    r["u"] = utility(r["comp"], r["ref"])
    r["p"] = privacy(r["prompt"], r["base"])

print("Scores computed for", len(results), "items.")





In [ ]:
#Inspect first few points
for level in ["original", "low", "high"]:
    print("\nLevel:", level)
    subset = [r for r in results if r["level"] == level]
    for r in subset[:3]:
        print(
            "p =", round(r["p"], 3),
            "u =", round(r["u"], 3)
        )


In [ ]:
#check average scores
def avg(xs):
    return sum(xs) / len(xs) if xs else 0.0

for level in ["original", "low", "high"]:
    vals = [r for r in results if r["level"] == level]
    avg_p = avg([r["p"] for r in vals])
    avg_u = avg([r["u"] for r in vals])
    print(
    level,
    "avg privacy:", round(avg_p, 3),
    "avg utility:", round(avg_u, 3)
    )


In [ ]:
# Collect scores per obfuscation level for the plot
orig_p = [r["p"] for r in results if r["level"] == "original"]
orig_u = [r["u"] for r in results if r["level"] == "original"]
low_p = [r["p"] for r in results if r["level"] == "low"]
low_u = [r["u"] for r in results if r["level"] == "low"]
high_p = [r["p"] for r in results if r["level"] == "high"]
high_u = [r["u"] for r in results if r["level"] == "high"]

#Plot Privacy vs Utility


plt.figure(figsize=(8, 6))

# Each group in a different color
plt.scatter(orig_p, orig_u, label="original")
plt.scatter(low_p, low_u, label="low obfuscation")
plt.scatter(high_p, high_u, label="high obfuscation")

plt.xlabel("Privacy score (normalized distance)")
plt.ylabel("Utility score (ROUGE-L style F1)")
plt.title("Privacy Utility tradeoff for code completion")
plt.legend()
plt.grid(True)
plt.savefig("privacy_utility.png", dpi=150, bbox_inches="tight")
plt.show()

# Analysis of utility privacy trade-off

The results show a clear privacy utility tradeoff. Original prompts have a privacy score of 0.0 and an average utility of 0.228, since the model sees the full, unmodified task description. Low obfuscation increases the average privacy score to 0.531 and reduces the average utility to 0.078, which indicates that partial masking of identifiers lowers completion quality. High obfuscation reaches an average privacy score of 0.889 and drops the utility to 0.008, because most of the original information is removed. Overall, as the obfuscation strength increases, privacy improves steadily while utility declines sharply.